In [7]:
from datetime import datetime
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

# ==========================================================
# CONEXION MONGODB
# ==========================================================
try:
    client = MongoClient("mongodb://bigdata_mongodb:27017/")
    db = client["proyecto_bigdata"]
    coleccion = db["alojamientos"]
    print("Conexion a MongoDB exitosa!")
except Exception as e:
    print(f"Error conectando a MongoDB: {e}")

# ==========================================================
# CIUDADES A SCRAPEAR (CHILE)
# ==========================================================
ciudades = [
    "Arica",
    "Iquique",
    "Calama",
    "Antofagasta",
    "Copiapo",
    "La Serena",
    "Valparaiso",
    "Vina del Mar",
    "Santiago",
    "Rancagua",
    "Talca",
    "Chillan",
    "Concepcion",
    "Temuco",
    "Valdivia",
    "Puerto Varas",
    "Puerto Montt",
    "Coyhaique",
    "Puerto Natales",
    "Punta Arenas"
]

# ==========================================================
# FUNCIONES AUXILIARES
# ==========================================================
def limpiar_precio(texto):
    numeros = ""
    for c in texto:
        if c.isdigit():
            numeros += c

    if not numeros:
        return 0.0

    return float(numeros)


def configurar_driver():
    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-blink-features=AutomationControlled")

    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )

    return driver


def obtener_zona(ciudad):
    norte = [
        "Arica", "Iquique", "Calama",
        "Antofagasta", "Copiapo", "La Serena"
    ]

    centro = [
        "Valparaiso", "Vina del Mar",
        "Santiago", "Rancagua", "Talca"
    ]

    if ciudad in norte:
        return "Norte"
    elif ciudad in centro:
        return "Centro"
    else:
        return "Sur"


# ==========================================================
# SCRAPER AIRBNB
# ==========================================================
def scraper_airbnb(ciudad):

    url = (
        f"https://www.airbnb.cl/s/{ciudad.replace(' ', '-')}"
        f"--Chile/homes"
    )

    print("=" * 50)
    print("Ciudad:", ciudad)
    print("=" * 50)

    driver = None

    try:
        driver = configurar_driver()
        driver.get(url)

        time.sleep(6)

        alojamientos = driver.find_elements(
            By.CSS_SELECTOR,
            'div[itemprop="itemListElement"]'
        )

        print("Resultados encontrados:", len(alojamientos))

        guardados = 0

        for i, item in enumerate(alojamientos):

            try:
                nombre = item.text.split("\n")[0].strip()

                if not nombre:
                    continue

                texto = item.text
                precio = limpiar_precio(texto)

                registro = {
                    "nombre_hotel": nombre,
                    "precio_noche": precio,
                    "ciudad": ciudad,
                    "zona_geografica": obtener_zona(ciudad),
                    "plataforma": "airbnb.cl",
                    "fecha_captura": datetime.now(),
                    "integrante": "matias-gonzalez",
                    "grupo": "G5_Turismo_Hoteleria"
                }

                coleccion.update_one(
                    {
                        "nombre_hotel": nombre,
                        "ciudad": ciudad,
                        "plataforma": "airbnb.cl"
                    },
                    {"$set": registro},
                    upsert=True
                )

                guardados += 1

                print(f"[{i+1}] Guardado -> {nombre[:45]}")

            except:
                continue

        print(f"Total guardados en {ciudad}: {guardados}")
        return guardados

    except Exception as e:
        print("Error:", e)
        return 0

    finally:
        if driver:
            driver.quit()


# ==========================================================
# EJECUCION GENERAL
# ==========================================================
total_antes = coleccion.count_documents({"plataforma": "airbnb.cl"})
print("Registros antes:", total_antes)

total_nuevos = 0

for ciudad in ciudades:

    nuevos = scraper_airbnb(ciudad)
    total_nuevos += nuevos

    print("Esperando 10 segundos...\n")
    time.sleep(10)

total_despues = coleccion.count_documents({"plataforma": "airbnb.cl"})

print("=" * 50)
print("SCRAPING COMPLETADO")
print("=" * 50)
print("Antes:", total_antes)
print("Ahora:", total_despues)
print("Nuevos:", total_despues - total_antes)
print("=" * 50)

Conexion a MongoDB exitosa!
Registros antes: 0
Ciudad: Arica
Resultados encontrados: 18
[1] Guardado -> Favorito entre huéspedes
[2] Guardado -> Condo en Arica
[3] Guardado -> Favorito entre huéspedes
[4] Guardado -> Apartamento en Arica
[5] Guardado -> Apartamento en Arica
[6] Guardado -> Favorito entre huéspedes
[7] Guardado -> Superanfitrión
[8] Guardado -> Favorito entre huéspedes
[9] Guardado -> Favorito entre huéspedes preferido
[10] Guardado -> Favorito entre huéspedes
[11] Guardado -> Favorito entre huéspedes
[12] Guardado -> Favorito entre huéspedes
[13] Guardado -> Favorito entre huéspedes preferido
[14] Guardado -> Favorito entre huéspedes
[15] Guardado -> Superanfitrión
[16] Guardado -> Apartamento en Arica
[17] Guardado -> Apartamento en Arica
[18] Guardado -> Superanfitrión
Total guardados en Arica: 18
Esperando 10 segundos...

Ciudad: Iquique
Resultados encontrados: 18
[1] Guardado -> Favorito entre huéspedes preferido
[2] Guardado -> Favorito entre huéspedes preferido
T

In [8]:
from datetime import datetime
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
import time
import re

# ==========================================================
# MONGODB
# ==========================================================
client = MongoClient("mongodb://bigdata_mongodb:27017/")
db = client["proyecto_bigdata"]
coleccion = db["alojamientos"]

print("Conexion MongoDB exitosa!")

# ==========================================================
# CIUDADES
# ==========================================================
ciudades = [
    "Arica","Iquique","Calama","Antofagasta","Copiapo",
    "La Serena","Valparaiso","Vina del Mar","Santiago",
    "Rancagua","Talca","Chillan","Concepcion","Temuco",
    "Valdivia","Puerto Varas","Puerto Montt","Coyhaique",
    "Puerto Natales","Punta Arenas"
]

# ==========================================================
# DRIVER
# ==========================================================
def iniciar_driver():
    op = Options()
    op.add_argument("--headless")
    op.add_argument("--no-sandbox")
    op.add_argument("--disable-dev-shm-usage")
    op.add_argument("--disable-blink-features=AutomationControlled")

    op.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=op
    )

    return driver


# ==========================================================
# LIMPIAR PRECIO
# ==========================================================
def limpiar_precio(txt):

    nums = re.findall(r"\d+", txt)

    if nums:
        return float("".join(nums))

    return 0.0


# ==========================================================
# ZONA
# ==========================================================
def zona(ciudad):

    norte = [
        "Arica","Iquique","Calama",
        "Antofagasta","Copiapo","La Serena"
    ]

    centro = [
        "Valparaiso","Vina del Mar",
        "Santiago","Rancagua","Talca"
    ]

    if ciudad in norte:
        return "Norte"
    elif ciudad in centro:
        return "Centro"
    else:
        return "Sur"


# ==========================================================
# SCROLL
# ==========================================================
def scroll_airbnb(driver):

    last = 0

    for i in range(8):

        driver.execute_script(
            "window.scrollTo(0, document.body.scrollHeight);"
        )

        time.sleep(3)

        new = driver.execute_script(
            "return document.body.scrollHeight"
        )

        if new == last:
            break

        last = new


# ==========================================================
# SCRAPER PRO
# ==========================================================
def scrapear_ciudad(ciudad):

    print("="*60)
    print("Ciudad:", ciudad)
    print("="*60)

    url = f"https://www.airbnb.cl/s/{ciudad.replace(' ','-')}--Chile/homes"

    driver = iniciar_driver()

    try:

        driver.get(url)

        time.sleep(6)

        scroll_airbnb(driver)

        cards = driver.find_elements(
            By.CSS_SELECTOR,
            'div[itemprop="itemListElement"]'
        )

        print("Cards detectadas:", len(cards))

        guardados = 0

        for i, card in enumerate(cards):

            try:

                texto = card.text.strip()

                lineas = texto.split("\n")

                if len(lineas) < 2:
                    continue

                # ==================================================
                # NOMBRE REAL
                # ==================================================
                nombre = ""

                for linea in lineas:

                    if (
                        "Favorito" not in linea
                        and "Superanfitrión" not in linea
                        and "$" not in linea
                        and "CLP" not in linea
                        and len(linea) > 8
                    ):
                        nombre = linea
                        break

                if nombre == "":
                    continue

                # ==================================================
                # PRECIO
                # ==================================================
                precio = limpiar_precio(texto)

                # ==================================================
                # URL
                # ==================================================
                try:
                    link = card.find_element(
                        By.TAG_NAME,"a"
                    ).get_attribute("href")
                except:
                    link = url

                # ==================================================
                # REGISTRO
                # ==================================================
                registro = {
                    "nombre_hotel": nombre,
                    "precio_noche": precio,
                    "ciudad": ciudad,
                    "zona_geografica": zona(ciudad),
                    "plataforma": "airbnb.cl",
                    "url_origen": link,
                    "fecha_captura": datetime.now(),
                    "integrante": "matias-gonzalez",
                    "grupo": "G5_Turismo_Hoteleria"
                }

                coleccion.update_one(
                    {
                        "nombre_hotel": nombre,
                        "ciudad": ciudad,
                        "plataforma": "airbnb.cl"
                    },
                    {"$set": registro},
                    upsert=True
                )

                guardados += 1

                print(f"[{i+1}] {nombre[:45]}")

            except:
                continue

        print("Guardados:", guardados)

        return guardados

    except Exception as e:

        print("Error:", e)
        return 0

    finally:
        driver.quit()


# ==========================================================
# EJECUCION GENERAL
# ==========================================================
antes = coleccion.count_documents({"plataforma":"airbnb.cl"})

print("Antes:", antes)

total = 0

for ciudad in ciudades:

    total += scrapear_ciudad(ciudad)

    print("Esperando...\n")
    time.sleep(8)

despues = coleccion.count_documents({"plataforma":"airbnb.cl"})

print("="*60)
print("SCRAPING FINALIZADO")
print("="*60)
print("Antes:", antes)
print("Ahora:", despues)
print("Nuevos:", despues - antes)
print("="*60)

Conexion MongoDB exitosa!
Antes: 52
Ciudad: Arica
Cards detectadas: 18
[1] Apartamento en Arica
[2] Condo en Arica
[3] Apartamento en Arica
[4] Apartamento en Arica
[5] Apartamento en Arica
[6] Condo en Arica
[7] Condo en Arica
[8] Habitación en Arica
[9] Apartamento en Arica
[10] Apartamento en Arica
[11] Bed and breakfast en Arica
[12] Condo en Arica
[13] Apartamento en Arica
[14] Habitación en Arica
[15] Condo en Arica
[16] Apartamento en Arica
[17] Apartamento en Arica
[18] Alojamiento vacacional en Arica
Guardados: 18
Esperando...

Ciudad: Iquique
Cards detectadas: 18
[1] Apartamento en Iquique
[2] Alojamiento en Iquique
[3] Apartamento en Iquique
[4] Apartamento en Iquique
[5] Apartamento en Iquique
[6] Apartamento en Iquique
[7] Apartamento en Iquique
[8] Apartamento en Iquique
[9] Apartamento en Iquique
[10] Apartamento en Iquique
[11] Apartamento en Iquique
[12] Loft en Iquique
[13] Apartamento en Iquique
[14] Apartamento en Iquique
[15] Apartamento en Iquique
[16] Apartamento

In [9]:
from datetime import datetime
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time
import re

# ==========================================
# CONEXION MONGODB
# ==========================================
client = MongoClient("mongodb://bigdata_mongodb:27017/")
db = client["proyecto_bigdata"]
coleccion = db["alojamientos"]

print("Conexion MongoDB exitosa!")

# ==========================================
# CIUDADES
# ==========================================
ciudades = [
    "Arica","Iquique","Calama","Antofagasta","Copiapo",
    "La Serena","Valparaiso","Vina del Mar","Santiago",
    "Rancagua","Talca","Chillan","Concepcion","Temuco",
    "Valdivia","Puerto Varas","Puerto Montt","Coyhaique",
    "Puerto Natales","Punta Arenas"
]

# ==========================================
# CONFIG DRIVER
# ==========================================
def configurar_driver():
    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )
    return driver

# ==========================================
# EXTRAER PRECIO
# ==========================================
def extraer_precio(texto):
    nums = re.findall(r'\d+', texto.replace(".", ""))
    if nums:
        return int("".join(nums))
    return 0

# ==========================================
# ZONA GEOGRAFICA
# ==========================================
def obtener_zona(ciudad):
    if ciudad in ["Arica","Iquique","Calama","Antofagasta","Copiapo"]:
        return "Norte"
    elif ciudad in ["La Serena","Valparaiso","Vina del Mar","Santiago","Rancagua"]:
        return "Centro"
    elif ciudad in ["Talca","Chillan","Concepcion","Temuco","Valdivia"]:
        return "Sur"
    else:
        return "Austral"

# ==========================================
# SCRAPER
# ==========================================
def scraper_airbnb(ciudad):

    url = f"https://www.airbnb.cl/s/{ciudad.replace(' ','-')}/homes?adults=2"

    print("="*60)
    print("Ciudad:", ciudad)
    print("="*60)

    driver = configurar_driver()
    driver.get(url)

    time.sleep(6)

    # ======================================
    # SCROLL PARA MAS DE 18 REGISTROS
    # ======================================
    last_height = driver.execute_script("return document.body.scrollHeight")

    for _ in range(12):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            break

        last_height = new_height

    # ======================================
    # CAPTURAR CARDS
    # ======================================
    cards = driver.find_elements(By.CSS_SELECTOR, '[itemprop="itemListElement"]')

    print("Cards detectadas:", len(cards))

    guardados = 0

    for i, card in enumerate(cards):

        try:
            texto = card.text.strip()

            if texto == "":
                continue

            lineas = texto.split("\n")

            nombre = lineas[0]
            precio = 0
            puntuacion = 0
            estrellas = 0
            tipo_habitacion = "No especificado"

            for linea in lineas:

                if "$" in linea:
                    precio = extraer_precio(linea)

                if re.search(r'\d\.\d', linea):
                    try:
                        puntuacion = float(
                            re.search(r'\d\.\d', linea).group()
                        )
                    except:
                        pass

                if ("Apartamento" in linea or
                    "Habitación" in linea or
                    "Condo" in linea or
                    "Casa" in linea or
                    "Hotel" in linea):
                    tipo_habitacion = linea

            # estrellas aproximadas
            if puntuacion >= 4.8:
                estrellas = 5
            elif puntuacion >= 4.5:
                estrellas = 4
            elif puntuacion >= 4.0:
                estrellas = 3
            elif puntuacion > 0:
                estrellas = 2

            # ======================================
            # REGISTRO COMPLETO
            # ======================================
            registro = {

                "integrante": "matias-gonzalez",
                "grupo": "G5_Turismo_Hoteleria",

                "nombre": nombre,
                "nombre_hotel": nombre,

                "ciudad": ciudad,
                "zona_geografica": obtener_zona(ciudad),

                "precio": precio,
                "precio_clp": precio,

                "estrellas": estrellas,
                "puntuacion": puntuacion,

                "adultos": 2,
                "adultos_habitacion": 2,

                "noches": 3,

                "tipo_habitacion": tipo_habitacion,

                "url_origen": url,
                "plataforma": "airbnb.cl",

                "fecha": datetime.now(),
                "fecha_captura": datetime.now()
            }

            coleccion.insert_one(registro)

            guardados += 1

            print(f"[{i+1}] {nombre} | ${precio}")

        except:
            continue

    print("Guardados:", guardados)

    driver.quit()

# ==========================================
# MAIN
# ==========================================
antes = coleccion.count_documents({"plataforma":"airbnb.cl"})
print("Antes:", antes)

for ciudad in ciudades:
    scraper_airbnb(ciudad)
    print("Esperando...")
    time.sleep(5)

despues = coleccion.count_documents({"plataforma":"airbnb.cl"})

print("="*60)
print("SCRAPING FINALIZADO")
print("="*60)
print("Antes:", antes)
print("Ahora:", despues)
print("Nuevos:", despues - antes)
print("="*60)

Conexion MongoDB exitosa!
Antes: 120
Ciudad: Arica
Cards detectadas: 18
[1] Apartamento en Arica | $1850005
[2] Favorito entre huéspedes preferido | $3727685
[3] Favorito entre huéspedes preferido | $3387205
[4] Favorito entre huéspedes | $1798725
[5] Favorito entre huéspedes | $1226415
[6] Favorito entre huéspedes | $1950575
[7] Favorito entre huéspedes preferido | $124743
[8] Favorito entre huéspedes preferido | $3737615
[9] Superanfitrión | $2044005
[10] Condo en Arica | $164688
[11] Favorito entre huéspedes | $86419
[12] Favorito entre huéspedes | $204400
[13] Favorito entre huéspedes | $3212015
[14] Favorito entre huéspedes | $3562405
[15] Favorito entre huéspedes | $1868805
[16] Favorito entre huéspedes | $2044005
[17] Favorito entre huéspedes | $3387205
[18] Apartamento en Arica | $2039795
Guardados: 18
Esperando...
Ciudad: Iquique
Cards detectadas: 18
[1] Favorito entre huéspedes | $2117565
[2] Apartamento en Iquique | $188632
[3] Favorito entre huéspedes preferido | $2482005
[